Types of Error:
1. Boundary Errors
2. Abbreviation Erros
3. Overlap/Ambiguity Error
4. False Positives
5. Rare Unseen Entities

In [2]:
from pathlib import Path

print(Path.cwd())

d:\Research Work\Projects and Courses\Medical_Text_Analysis\notebooks


In [9]:
import os

print(os.listdir("../models/spacy"))

['biomedical_ner']


In [14]:
model_path = Path("../models/spacy/biomedical_ner")
print(model_path.exists())

True


In [12]:
from pathlib import Path

print(Path("../models").resolve())
print(Path("../models/spacy").resolve())
print(Path("../models/spacy/biomedical_ner").resolve())

print(Path("../models").exists())
print(Path("../models/spacy").exists())
print(Path("../models/spacy/biomedical_ner").exists())

D:\Research Work\Projects and Courses\Medical_Text_Analysis\models
D:\Research Work\Projects and Courses\Medical_Text_Analysis\models\spacy
D:\Research Work\Projects and Courses\Medical_Text_Analysis\models\spacy\biomedical_ner
True
True
True


In [15]:
import spacy

nlp = spacy.load("../models/spacy/biomedical_ner")

In [24]:
import json

test_data_path = "../data/preprocessed/spacy/test_spacy.json"

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

In [30]:
print(test_data[0][1]["entities"])

[[0, 10, 'CHEMICAL'], [24, 32, 'DISEASE']]


In [ ]:
#Extracting Gold Entities
gold_entities = []
predicted_entities = []

for i in range(len(test_data)):
    gold_entities.append(test_data[i][1]["entities"])

print(gold_entities[:10])

[[[0, 10, 'CHEMICAL'], [24, 32, 'DISEASE']], [], [[0, 10, 'CHEMICAL'], [103, 109, 'DISEASE']], [[101, 109, 'DISEASE'], [173, 183, 'CHEMICAL']], [[35, 45, 'CHEMICAL'], [59, 67, 'DISEASE'], [132, 142, 'CHEMICAL']], [[24, 34, 'CHEMICAL']], [[26, 36, 'CHEMICAL']], [[0, 12, 'CHEMICAL'], [21, 32, 'DISEASE'], [36, 42, 'CHEMICAL']], [[38, 50, 'CHEMICAL'], [53, 56, 'CHEMICAL'], [62, 68, 'CHEMICAL']], [[3, 9, 'CHEMICAL'], [27, 39, 'CHEMICAL']]]


In [35]:
doc = nlp("Apirin help bro")
print(doc)

Apirin help bro


In [36]:
for i in range(len(test_data)):
    entities = []
    text = test_data[i][0]
    doc = nlp(text)

    for ent in doc.ents:
        entities.append([
            ent.start_char,
            ent.end_char,
            ent.label_
        ])
    
    predicted_entities.append(entities)

print(predicted_entities[:10])

[[[0, 10, 'CHEMICAL'], [24, 32, 'DISEASE']], [], [[0, 10, 'CHEMICAL'], [16, 25, 'CHEMICAL']], [[101, 109, 'DISEASE'], [173, 183, 'CHEMICAL']], [[35, 45, 'CHEMICAL'], [59, 67, 'DISEASE'], [132, 142, 'CHEMICAL']], [[24, 34, 'CHEMICAL']], [[26, 36, 'CHEMICAL']], [[0, 12, 'CHEMICAL'], [21, 32, 'DISEASE'], [36, 42, 'CHEMICAL']], [[38, 50, 'CHEMICAL'], [62, 68, 'CHEMICAL']], [[3, 9, 'CHEMICAL'], [27, 39, 'CHEMICAL']]]


In [52]:
wrong_predictions = []

boundary_error = 0
overlap_error = 0
false_negative = 0
false_positive = 0


def overlap(start1, end1, start2, end2):
    return max(start1, start2) < min(end1, end2)


for sentence, gold, pred in zip(texts, gold_entities, predicted_entities):

    gold_set = set(tuple(x) for x in gold)
    pred_set = set(tuple(x) for x in pred)

    exact_matches = gold_set & pred_set

    unmatched_gold = gold_set - exact_matches
    unmatched_pred = pred_set - exact_matches

    used_predictions = set()

    # Check each gold entity
    for gold_entity in unmatched_gold:

        g_start, g_end, g_label = gold_entity

        found_match = False

        for pred_entity in unmatched_pred:

            p_start, p_end, p_label = pred_entity

            if overlap(g_start, g_end, p_start, p_end):

                found_match = True
                used_predictions.add(pred_entity)

                error_record = {
                    "sentence": sentence,
                    "gold_entity": sentence[g_start:g_end],
                    "pred_entity": sentence[p_start:p_end],
                    "gold_label": g_label,
                    "pred_label": p_label,
                    "error": ""
                }

                if g_label == p_label:

                    boundary_error += 1
                    error_record["error"] = "Boundary Error"

                else:

                    overlap_error += 1
                    error_record["error"] = "Overlap/Ambiguity Error"

                wrong_predictions.append(error_record)

                break

        if not found_match:

            false_negative += 1

            wrong_predictions.append(
                {
                    "sentence": sentence,
                    "gold_entity": sentence[g_start:g_end],
                    "pred_entity": None,
                    "gold_label": g_label,
                    "pred_label": None,
                    "error": "False Negative"
                }
            )

    # Remaining predictions = FP
    for pred_entity in unmatched_pred:

        if pred_entity not in used_predictions:

            p_start, p_end, p_label = pred_entity

            false_positive += 1

            wrong_predictions.append(
                {
                    "sentence": sentence,
                    "gold_entity": None,
                    "pred_entity": sentence[p_start:p_end],
                    "gold_label": None,
                    "pred_label": p_label,
                    "error": "False Positive"
                }
            )

In [53]:
print("Boundary:", boundary_error)
print("Overlap:", overlap_error)
print("FN:", false_negative)
print("FP:", false_positive)

Boundary: 608
Overlap: 173
FN: 1682
FP: 719


In [55]:
from pprint import pprint

pprint(wrong_predictions[:5])

[{'error': 'False Negative',
  'gold_entity': 'ulcers',
  'gold_label': 'DISEASE',
  'pred_entity': None,
  'pred_label': None,
  'sentence': 'Famotidine is a histamine H2 - receptor antagonist used in '
              'inpatient settings for prevention of stress ulcers and is '
              'showing increasing popularity because of its low cost .'},
 {'error': 'False Positive',
  'gold_entity': None,
  'gold_label': None,
  'pred_entity': 'histamine',
  'pred_label': 'CHEMICAL',
  'sentence': 'Famotidine is a histamine H2 - receptor antagonist used in '
              'inpatient settings for prevention of stress ulcers and is '
              'showing increasing popularity because of its low cost .'},
 {'error': 'False Negative',
  'gold_entity': 'IDM',
  'gold_label': 'CHEMICAL',
  'pred_entity': None,
  'pred_label': None,
  'sentence': 'After a single oral dose of 4 mg / kg indomethacin ( IDM ) to '
              'sodium and volume depleted rats plasma renin activity ( PRA ) '
      

In [57]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

In [58]:
from src.transformer_pipeline.dataset import get_tokenized_dataset, tokenizer

text = "Aspirin help me."
encoding = tokenizer(
    text,
    return_offsets_mapping=True,
    truncation=True
)

d:\Research Work\Projects and Courses\Medical_Text_Analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [59]:
print(encoding)

{'input_ids': [101, 1249, 8508, 4854, 1494, 1143, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1], 'offset_mapping': [(0, 0), (0, 2), (2, 4), (4, 7), (8, 12), (13, 15), (15, 16), (0, 0)]}


In [60]:
dataset = get_tokenized_dataset()

print(dataset)

Map: 100%|██████████| 5865/5865 [00:02<00:00, 2618.23 examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5330
    })
    test: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5865
    })
})


In [71]:
print(dataset["test"]["tokens"])
print(dataset["test"]["labels"])
print(dataset["test"]["tags"])

[['Famotidine', '-', 'associated', 'delirium', '.'], ['A', 'series', 'of', 'six', 'cases', '.'], ['Famotidine', 'is', 'a', 'histamine', 'H2', '-', 'receptor', 'antagonist', 'used', 'in', 'inpatient', 'settings', 'for', 'prevention', 'of', 'stress', 'ulcers', 'and', 'is', 'showing', 'increasing', 'popularity', 'because', 'of', 'its', 'low', 'cost', '.'], ['Although', 'all', 'of', 'the', 'currently', 'available', 'H2', '-', 'receptor', 'antagonists', 'have', 'shown', 'the', 'propensity', 'to', 'cause', 'delirium', ',', 'only', 'two', 'previously', 'reported', 'cases', 'have', 'been', 'associated', 'with', 'famotidine', '.'], ['The', 'authors', 'report', 'on', 'six', 'cases', 'of', 'famotidine', '-', 'associated', 'delirium', 'in', 'hospitalized', 'patients', 'who', 'cleared', 'completely', 'upon', 'removal', 'of', 'famotidine', '.'], ['The', 'pharmacokinetics', 'of', 'famotidine', 'are', 'reviewed', ',', 'with', 'no', 'change', 'in', 'its', 'metabolism', 'in', 'the', 'elderly', 'populati

In [67]:
print(dataset["train"][0]["tags"])

[1, 0, 0, 0, 0, 0, 1, 0]


In [70]:
texts = []
gold_entities = []

for text, annotation in test_data:
    texts.append(text)
    gold_entities.append(annotation["entities"])

print(texts)
print(gold_entities)

['Famotidine - associated delirium .', 'A series of six cases .', 'Famotidine is a histamine H2 - receptor antagonist used in inpatient settings for prevention of stress ulcers and is showing increasing popularity because of its low cost .', 'Although all of the currently available H2 - receptor antagonists have shown the propensity to cause delirium , only two previously reported cases have been associated with famotidine .', 'The authors report on six cases of famotidine - associated delirium in hospitalized patients who cleared completely upon removal of famotidine .', 'The pharmacokinetics of famotidine are reviewed , with no change in its metabolism in the elderly population seen .', 'The implications of using famotidine', 'Indomethacin induced hypotension in sodium and volume depleted rats .', 'After a single oral dose of 4 mg / kg indomethacin ( IDM ) to sodium and volume depleted rats plasma renin activity ( PRA ) and systolic blood pressure fell significantly within four hours

In [87]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)
import torch
from src.transformer_pipeline.config import ID2LABEL

tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-v1.1"
)
model = AutoModelForTokenClassification.from_pretrained(
    "../models/biobert/model_7epoch"
)

model.eval()

def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop("offset_mapping")

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(
        outputs.logits,
        dim=2
    )

    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    labels = [
        ID2LABEL[label.item()]
        for label in predictions[0]
    ]

    offsets = offset_mapping[0]

    return list(zip(tokens, labels, offsets))

def extract_entities(text):

    predictions = predict(text)

    entities = []

    current_start = None
    current_end = None
    current_label = None

    for token, label, offset in predictions:

        start, end = offset.tolist()

        if token in ["[CLS]", "[SEP]"]:
            continue

        if start == end:
            continue

        if label.startswith("B-"):

            if current_start is not None:

                entities.append(
                    [
                        current_start,
                        current_end,
                        current_label
                    ]
                )

            current_start = start
            current_end = end
            current_label = label[2:]

        elif (
            label.startswith("I-")
            and current_label == label[2:]
        ):

            current_end = end

        else:

            if current_start is not None:

                entities.append(
                    [
                        current_start,
                        current_end,
                        current_label
                    ]
                )

            current_start = None
            current_end = None
            current_label = None

    if current_start is not None:

        entities.append(
            [
                current_start,
                current_end,
                current_label
            ]
        )

    return entities

predicted_entities = []

for i in range(len(test_data)):
    text = test_data[i][0]
    predicted_entities.append(extract_entities(text))

print(predicted_entities)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 753.31it/s]


[[[0, 10, 'CHEMICAL'], [24, 32, 'DISEASE']], [], [[0, 10, 'CHEMICAL'], [16, 25, 'CHEMICAL'], [96, 102, 'DISEASE'], [103, 109, 'DISEASE']], [[101, 109, 'DISEASE'], [173, 183, 'CHEMICAL']], [[35, 45, 'CHEMICAL'], [59, 67, 'DISEASE'], [132, 142, 'CHEMICAL']], [[24, 34, 'CHEMICAL']], [[26, 36, 'CHEMICAL']], [[0, 12, 'CHEMICAL'], [21, 32, 'DISEASE'], [36, 42, 'CHEMICAL']], [[38, 50, 'CHEMICAL'], [53, 56, 'CHEMICAL'], [62, 68, 'CHEMICAL']], [[3, 9, 'CHEMICAL'], [27, 39, 'CHEMICAL']], [[7, 19, 'CHEMICAL'], [37, 50, 'CHEMICAL'], [138, 149, 'CHEMICAL'], [160, 166, 'CHEMICAL']], [[13, 24, 'DISEASE'], [25, 37, 'DISEASE'], [49, 59, 'CHEMICAL'], [64, 76, 'CHEMICAL']], [[0, 24, 'DISEASE'], [27, 30, 'DISEASE'], [59, 77, 'DISEASE'], [80, 83, 'DISEASE']], [[82, 85, 'DISEASE']], [[41, 67, 'DISEASE'], [84, 96, 'CHEMICAL'], [114, 117, 'DISEASE']], [[43, 46, 'DISEASE'], [58, 68, 'CHEMICAL']], [[57, 67, 'CHEMICAL'], [89, 92, 'DISEASE']], [[35, 50, 'CHEMICAL'], [51, 60, 'DISEASE'], [64, 79, 'CHEMICAL']], [[9

In [89]:
from pprint import pprint

def overlap(start1, end1, start2, end2):
    return max(start1, start2) < min(end1, end2)


def analyze_errors(texts, gold_entities, predicted_entities):

    stats = {
        "CHEMICAL": {
            "boundary_errors": 0,
            "overlap_errors": 0,
            "false_positives": 0,
            "false_negatives": 0
        },
        "DISEASE": {
            "boundary_errors": 0,
            "overlap_errors": 0,
            "false_positives": 0,
            "false_negatives": 0
        }
    }

    error_examples = {
        "CHEMICAL": [],
        "DISEASE": []
    }

    for sentence, gold, pred in zip(
        texts,
        gold_entities,
        predicted_entities
    ):

        gold_set = set(tuple(entity) for entity in gold)
        pred_set = set(tuple(entity) for entity in pred)

        exact_matches = gold_set & pred_set

        unmatched_gold = gold_set - exact_matches
        unmatched_pred = pred_set - exact_matches

        used_predictions = set()

        # Analyze unmatched gold entities
        for gold_entity in unmatched_gold:

            g_start, g_end, g_label = gold_entity

            found_match = False

            for pred_entity in unmatched_pred:

                # Prevent one prediction from matching multiple gold entities
                if pred_entity in used_predictions:
                    continue

                p_start, p_end, p_label = pred_entity

                if overlap(
                    g_start,
                    g_end,
                    p_start,
                    p_end
                ):

                    found_match = True
                    used_predictions.add(pred_entity)

                    error_record = {
                        "sentence": sentence,
                        "gold_entity": sentence[g_start:g_end],
                        "pred_entity": sentence[p_start:p_end],
                        "gold_label": g_label,
                        "pred_label": p_label,
                        "error": ""
                    }

                    if g_label == p_label:

                        stats[g_label]["boundary_errors"] += 1
                        error_record["error"] = "Boundary Error"

                    else:

                        stats[g_label]["overlap_errors"] += 1
                        error_record["error"] = (
                            "Overlap/Ambiguity Error"
                        )

                    error_examples[g_label].append(
                        error_record
                    )

                    break

            # False Negative
            if not found_match:

                stats[g_label]["false_negatives"] += 1

                error_examples[g_label].append(
                    {
                        "sentence": sentence,
                        "gold_entity": sentence[g_start:g_end],
                        "pred_entity": None,
                        "gold_label": g_label,
                        "pred_label": None,
                        "error": "False Negative"
                    }
                )

        # False Positives
        for pred_entity in unmatched_pred:

            if pred_entity in used_predictions:
                continue

            p_start, p_end, p_label = pred_entity

            if p_label not in stats:
                continue

            stats[p_label]["false_positives"] += 1

            error_examples[p_label].append(
                {
                    "sentence": sentence,
                    "gold_entity": None,
                    "pred_entity": sentence[p_start:p_end],
                    "gold_label": None,
                    "pred_label": p_label,
                    "error": "False Positive"
                }
            )

    return {
        "summary": stats,
        "examples": error_examples
    }

pprint(analyze_errors(texts, gold_entities, predicted_entities))

{'examples': {'CHEMICAL': [{'error': 'False Positive',
                            'gold_entity': None,
                            'gold_label': None,
                            'pred_entity': 'histamine',
                            'pred_label': 'CHEMICAL',
                            'sentence': 'Famotidine is a histamine H2 - '
                                        'receptor antagonist used in inpatient '
                                        'settings for prevention of stress '
                                        'ulcers and is showing increasing '
                                        'popularity because of its low cost .'},
                           {'error': 'False Negative',
                            'gold_entity': 'corticosteroid',
                            'gold_label': 'CHEMICAL',
                            'pred_entity': None,
                            'pred_label': None,
                            'sentence': 'Moderate to high dose corticosteroid '
  